# Modules and Model Construction

The multilayer perceptron of that section had four
parameter arrays, and we could name each one by hand. The models in the rest
of this book have many more. A residual network for images stacks dozens of
layers, and the language model Llama 3 8B
[@Grattafiori.Dubey.Jauhri.ea.2024], with 8 billion parameters,
consists of 32 identical blocks and stores its parameters in 291 arrays.
Every one of them must
reach the optimizer, be written to a checkpoint, and move to the GPU. An
array that is left out raises no error: it is simply never trained, or never
saved.

Deep learning frameworks solve this with one abstraction, the *module*. A
module is an object that owns parameters, owns other modules (its
*children*), and defines a forward computation from inputs to outputs.
Layers, groups of layers, and whole models are all modules, so a model is a
tree of modules, as in the figure. The framework finds every
parameter by walking this tree, provided that each child has been
*registered* with its parent. This section shows how registration works, how
it fails, and how to write the forward computation of a module yourself. The
rest of the chapter uses the same walk for other tasks: counting the
parameters (that section), initializing them
(that section), saving them (that section), and
moving them to a GPU (that section).

![The module tree of the residual MLP that we build at the end of this section. Each edge carries the name under which the parent registered the child, and the path of names from the root to a parameter is the name of that parameter, here `1.body.0.weight`. Blocks 2 to 4 repeat block 1. PyTorch and Gluon join the names with dots; JAX lists the same path as a tuple and keeps the children of a sequential container under `layers`. Gluon folds the ReLU into the first dense layer, so its second dense layer is child 1 of the body; Keras names the children after their classes.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-module-names.svg)

In [1]:
from dataclasses import dataclass
import tensorflow as tf

## The Module Abstraction

### Sequential Containers and the Registry

Every model since that section has been a module: `d2l.Module`
is a subclass of the framework's module class. The quickest way to build a
module from layers is a *sequential container*, a module that applies its
children one after the other. Here it holds the MLP of
that section, with a hidden layer of 256 ReLU units and
10 outputs, applied to a minibatch of two examples with 20 features each.

In Keras the module class is `tf.keras.layers.Layer`; a `tf.keras.Model` is
a layer with training methods added. The container `tf.keras.Sequential`
takes a list of layers, and the argument `activation` attaches the ReLU to
the first `Dense` layer.

In [2]:
net = tf.keras.Sequential([tf.keras.layers.Dense(256, activation='relu'),
                           tf.keras.layers.Dense(10)])

X = tf.random.uniform((2, 20))
net(X).shape

TensorShape([2, 10])

The container keeps its children in a *registry*, a table from names to
modules, and its forward computation runs through that table in order.

Keras keeps the children in a tracked list, `layers`. The names are
generated from the class names.

In [3]:
net.layers

[<Dense name=dense, built=True>, <Dense name=dense_1, built=True>]

Everything the framework does to a model as a whole walks this registry.
Collecting the parameters for the optimizer, writing them to a file, and
moving them to a GPU all visit the registered children, then their children,
and so on down to the parameters. A layer that is not registered is skipped
by all of them.

### Subclasses of the Module Class

A sequential container only expresses a chain. For any other wiring we
subclass the module class and write two methods: a constructor that creates
the children, and a forward method that applies them to the input.

In Keras the forward method is `call`. The `d2l.Module` of earlier chapters
defines `call` to run `forward`, which is why its subclasses wrote
`forward`; a plain Keras layer implements `call` directly.

Here is the same MLP once more:

In [4]:
class MLP(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.hidden = tf.keras.layers.Dense(256, activation='relu')
        self.out = tf.keras.layers.Dense(10)

    def call(self, X):
        return self.out(self.hidden(X))

In [5]:
net = MLP()
net(X).shape

TensorShape([2, 10])

Two details make this work. First, `self.hidden = ...` is not an ordinary
attribute assignment. The module class intercepts assignment (the Python
method `__setattr__`), and when the value is a module it enters the
registry under the attribute name, here `hidden` and `out`. That is all it
takes for both layers' parameters to reach the optimizer. Second, we wrote
no backward method: automatic differentiation derives the gradients from
whatever the forward method computes.

We call the model as `net(X)`, never as `net.call(X)`. Calling a layer runs
Keras's `__call__`, which on the first call creates the layer's variables
from the shape of the input (that section) and then runs `call`.

## Module Containers

### Writing a Container

A container has two jobs: register each child, and run the children in
order. Writing one ourselves shows both. The registration step is the one
that can go wrong.

In [6]:
class MySequential(tf.keras.Model):
    def __init__(self, *args):
        super().__init__()
        self.modules = args

    def call(self, X):
        for module in self.modules:
            X = module(X)
        return X

`self.modules = args` looks like an ordinary assignment. Keras's
`__setattr__` looks inside lists, tuples, and dictionaries, so both `Dense`
layers are tracked.

Our container works like the built-in one:

In [7]:
net = MySequential(tf.keras.layers.Dense(256, activation='relu'),
                   tf.keras.layers.Dense(10))
net(X).shape

TensorShape([2, 10])

### Layers in a Plain List

Now suppose we store the layers in an ordinary Python list. The forward
method can still loop over the list, but the list itself is not a module.
Does the framework still find the layers inside it?

In Keras the plain list is the ordinary way to hold a stack of layers,
because the attribute scan looks inside lists:

In [8]:
class ListMLP(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.blocks = [tf.keras.layers.Dense(256, activation='relu'),
                       tf.keras.layers.Dense(10)]

    def call(self, X):
        for block in self.blocks:
            X = block(X)
        return X

net = ListMLP()
net(X).shape, sum(int(tf.size(v)) for v in net.trainable_variables)

(TensorShape([2, 10]), 7946)

All 7,946 parameters are present: 20 · 256 + 256 in the hidden layer and
256 · 10 + 10 in the output layer. (We named the list `blocks` because
Keras reserves the attribute `layers` for its own tracked list.) The mistake
that Keras guards against is a different one: adding a layer to a model
whose variables already exist. Creating the variables fixes the structure
of the model, and a later assignment fails at once:

In [ ]:
try:
    net.head = tf.keras.layers.Dense(2)  # net is built: too late
except ValueError as e:
    print(str(e)[:70].rsplit(' ', 1)[0], '...')

The message states the rule: all state is created in `__init__` or in
`build`, never afterwards. Keras therefore needs no special list or
dictionary containers. Any layer reachable from an attribute is tracked,
and a late change to the structure fails when it is made, not when the
model silently ignores it.

## Forward Computation

The forward method is ordinary Python. It can call any tensor operation,
loop, branch, and combine intermediate results. This section uses that
freedom three times: for a residual connection, for a constant, and for a
loop whose length depends on the data.

### Residual Connections

The most important example in this book is the *residual connection*,
which adds the input of a block to its output. If $f$ is the *body* of the
block, the computation inside it, here two fully connected layers with a
ReLU between them, the block computes

$$\mathbf{y} = \mathbf{x} + f(\mathbf{x}).$$

The block outputs its input plus a correction $f(\mathbf{x})$. If $f$ is
zero, the block is the identity, so a stack of such blocks can pass its
input through unchanged, and each block only has to learn a correction to
its input. The networks of that section are built this way. No
framework provides this as a layer; it is one line of the forward method,
`X + self.body(X)`:

In [10]:
class ResidualBlock(tf.keras.layers.Layer):
    def __init__(self, num_hiddens, **kwargs):
        super().__init__(**kwargs)
        self.body = tf.keras.Sequential([
            tf.keras.layers.Dense(num_hiddens, activation='relu'),
            tf.keras.layers.Dense(num_hiddens)])

    def call(self, X):
        return X + self.body(X)

![A residual block. The input splits into the body, Linear, ReLU, Linear, and an identity path, and the two rejoin by addition.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-residual-block.svg)

the figure shows the wiring. The skip path also
changes the backward pass. The Jacobian of the block is
$\mathbf{I} + \mathbf{J}_f$, where $\mathbf{J}_f$ is the Jacobian of the
body, so the gradient that reaches $\mathbf{x}$ is the gradient at
$\mathbf{y}$ plus the part that flows through the body
[@He.Zhang.Ren.ea.2016*1]. The addition also has a consequence for
construction: $\mathbf{x}$ and $f(\mathbf{x})$ must have the same shape, so
a residual block has a single width, which we pass to the constructor.

Keras infers input widths when the layer is built, so the constraint falls
on the caller: an input with other than `num_hiddens` columns makes the
addition fail.

In [11]:
block = ResidualBlock(24)
block(tf.random.normal((2, 24))).shape

TensorShape([2, 24])

### Constants in the Forward Method

The forward method may also use values that are neither inputs nor
parameters. Suppose we want to damp the correction by a fixed factor
$\alpha = 0.5$, so that the block computes
$\mathbf{x} + \alpha f(\mathbf{x})$. The factor is part of the design, and
training must not change it:

In [12]:
class ScaledResidual(ResidualBlock):
    def __init__(self, num_hiddens, alpha=0.5, **kwargs):
        super().__init__(num_hiddens, **kwargs)
        self.alpha = tf.constant(alpha)  # Fixed by design, never trained

    def call(self, X):
        return X + self.alpha * self.body(X)

block = ScaledResidual(24)
block(tf.random.normal((2, 24)))
paths = [w.path for w in block.weights]
any('alpha' in path for path in paths), paths[:2]

(False,
 ['scaled_residual/sequential_2/dense_11/kernel',
  'scaled_residual/sequential_2/dense_11/bias'])

The factor enters the computation but is not a variable: `block.weights`
holds only the `Dense` variables, so the optimizer never changes it, which
is what we wanted. Not being a variable, it is not saved with the weights
either. Some state must travel with the model without being trained.
that section gives such state its registered place, a
non-trainable variable.

### Eager and Compiled Execution

A forward method is ordinary Python code, and so far it has also run like
ordinary Python, one line at a time. A framework can instead record the
computation once and replay the recording, which is faster but restricts
what the forward method may do.

Called directly, `call` runs as written, one operation at a time, a mode
called *eager execution*. The `Trainer` of that section
compiles each training step with `tf.function` instead: it runs the step
once with symbolic tensors that have shapes but no values, records the
operations, and from then on executes the recording. In a plain
`tf.function`, a converter called AutoGraph rewrites a Python loop whose
condition depends on the values of tensors into a graph loop. Keras does
not apply AutoGraph to the `call` method of a layer, however, so in a model
such a loop cannot be recorded:

In [13]:
class HalvingNet(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.proj = tf.keras.layers.Dense(24)

    def call(self, X):
        X = self.proj(X)
        while tf.reduce_sum(tf.abs(X)) > 1:  # Depends on the values of X
            X = X / 2
        return tf.reduce_sum(X)

net = HalvingNet()
print(net(X))
try:
    tf.function(net)(X)
except tf.errors.OperatorNotAllowedInGraphError as e:
    print(type(e).__name__)

tf.Tensor(0.113871574, shape=(), dtype=float32)
OperatorNotAllowedInGraphError


Eager execution runs the loop and returns a number. Under `tf.function` the
condition in `call` is a symbolic tensor without a value, and TensorFlow
refuses to turn it into a Python `bool`. Inside a layer, a loop that
depends on values must be written with `tf.while_loop`, which becomes part
of the recorded program. Loops over
the structure of the model, such as the loop over the children in
`MySequential`, cause no trouble: they run once while the step is recorded.

## Lazy Initialization

### Shapes from Data

Since that section, our Keras layers such as `Dense(256)`
have named only their output width. The kernel has shape
`(in_features, 256)`, and we never said what `in_features` is: the layer
reads it from the first minibatch.
Every Keras layer works this way. Here is the mechanism. Before the first
call the layer has no variables at all:

In [14]:
net = tf.keras.Sequential([tf.keras.layers.Dense(256, activation='relu'),
                           tf.keras.layers.Dense(10)])
net.weights

[]

Reading `net.layers[0].kernel` now would raise an error. The first call
runs each layer's `build`, which reads the input width from the minibatch
and creates the kernel. The output width of each layer is the input width
of the next, so the shapes cascade through the model:

In [15]:
net(X)
[w.shape for w in net.weights]

[TensorShape([20, 256]),
 TensorShape([256]),
 TensorShape([256, 10]),
 TensorShape([10])]

### Initialization after a Dry Run

Lazy layers spare us the arithmetic of input widths: in a deep model the
input width of each layer is the output width of the one before, and
working it out by hand is tedious and easy to get wrong. The cost is that
the parameters do not exist until a minibatch has passed through. Before
that, they cannot be counted, and an initializer whose scale depends on the
number of inputs, such as the Xavier initialization of
that section, cannot be applied. Initialization therefore follows
a *dry run*, one forward pass on a representative minibatch whose output is
discarded.

Keras attaches the initializer to the layer: each layer accepts a
`kernel_initializer`, and `build` uses it when it creates the kernel. The
dry run can also be replaced by `build` with the input shape, which
propagates shapes through the model without data (`None` stands for the
size of the minibatch). Xavier's uniform initializer is the Keras default,
`GlorotUniform`; we spell it out:

In [16]:
net = tf.keras.Sequential([
    tf.keras.layers.Dense(
        256, activation='relu',
        kernel_initializer=tf.keras.initializers.GlorotUniform()),
    tf.keras.layers.Dense(10)])
net.build((None, 20))
net.layers[0].kernel.shape

TensorShape([20, 256])

`build` created every variable and drew the first kernel from the Xavier
initializer, without any data.

## Models from a Config

### Configs and Builders

A trained model is a set of arrays plus the code that knows what to do with
them. The arrays alone do not say how to rebuild the model. Every width and
depth in this section so far was a literal typed into a constructor call,
and a checkpoint of the weights records such choices at best indirectly:
the shapes of the arrays reveal the widths, but nothing in them reveals a
choice without parameters, such as the activation function. The choices that
vary from one experiment to the next therefore belong in a small
*configuration* object, or *config*, that is logged with the results and
saved with the weights. The topology, meaning what a block computes and how
blocks connect, stays in code. Here a Python `dataclass` holds four fields,
and `build` turns them into a model:

In [17]:
@dataclass
class MLPConfig:
    num_inputs: int = 784
    num_hiddens: int = 256
    num_blocks: int = 4
    num_outputs: int = 10

def build(cfg: MLPConfig) -> tf.keras.Model:
    blocks = [ResidualBlock(cfg.num_hiddens) for _ in range(cfg.num_blocks)]
    return tf.keras.Sequential([tf.keras.Input((cfg.num_inputs,)),
                                tf.keras.layers.Dense(cfg.num_hiddens),
                                *blocks,
                                tf.keras.layers.Dense(cfg.num_outputs)])

One config gives one architecture: an input layer from `num_inputs` features to
the hidden width, `num_blocks` residual blocks, and an output layer. The
blocks pass through a plain Python list, which is safe here because
unpacking it into the sequential container registers each block. Every
layer receives both of its widths, so the model needs no dry run before its
parameters can be counted.

`tf.keras.Input` declares the input width, so `Sequential` builds the whole
model at construction. Keras names variables by the generated names of the
layers rather than by attribute names, but the path still runs from the
model through the block and its body to the variable:

In [18]:
net = build(MLPConfig())
[(w.path, tuple(w.shape)) for w in net.weights][:4]

[('sequential_9/dense_26/kernel', (784, 256)),
 ('sequential_9/dense_26/bias', (256,)),
 ('sequential_9/residual_block_1/sequential_5/dense_18/kernel', (256, 256)),
 ('sequential_9/residual_block_1/sequential_5/dense_18/bias', (256,))]

The model maps a minibatch of 784-dimensional inputs, the size of a
flattened Fashion-MNIST image, to 10 outputs:

In [19]:
net(tf.random.uniform((2, 784))).shape

TensorShape([2, 10])

### Counting Parameters

The config also tells us the size of the model before we build it. The
input layer has $784 \cdot 256 + 256 = 200{,}960$ parameters, each block
has two layers of $256 \cdot 256 + 256$, or $131{,}584$ together, and the
output layer has $256 \cdot 10 + 10 = 2{,}570$. The default model therefore
has

$$200{,}960 + 4 \cdot 131{,}584 + 2{,}570 = 729{,}866$$

parameters, most of them in the blocks. Rescaling the model is a change to
two fields, not an edit of the model code, and a walk over the parameters
confirms the count:

In [20]:
for cfg in (MLPConfig(), MLPConfig(num_hiddens=512, num_blocks=8)):
    print(f'num_hiddens={cfg.num_hiddens}, num_blocks={cfg.num_blocks}: '
          f'{build(cfg).count_params():,} parameters')

num_hiddens=256, num_blocks=4: 729,866 parameters
num_hiddens=512, num_blocks=8: 4,609,546 parameters


Doubling the width multiplies the parameters of each block by about four,
and doubling the number of blocks doubles their total, so the larger
config has more than six times as many parameters. The 8B model of the
introduction is described the same way: a config of
a dozen numbers, such as 32 blocks and a width of 4,096, and code that
turns it into a module tree.

A config records numbers, not code. Rebuilding a model from a checkpoint
also needs the same `build` function and the same block classes, which is
why that section saves the config inside the checkpoint next to
the weights.

Keras offers the same idea in its own form: every layer has a method
`get_config` that returns its constructor arguments, which Keras saves with
the model and uses to rebuild it, provided the layer classes are available.

We can now build a model of any size from modules, find all of its
parameters, and rebuild it from a few numbers. What we have not yet done is
look at the parameters themselves: which tensors a model holds, how much
memory they take, and how to share or freeze them. That is the subject of
that section.

## Summary

A module owns parameters, child modules, and a forward computation. Layers,
blocks, and whole models are modules, so a model is a tree, and the name of
each parameter is its path from the root. The framework finds parameters by
walking the registered children, so a child stored where the walk cannot
see it is never trained or saved. The forward method is ordinary Python, so
a residual connection is one line of it, although a compiled training step
forbids control flow that depends on the values of arrays. Lazy layers take
their input widths from the first minibatch, so their initialization follows
a dry run. A config of a few numbers, together with a `build` function,
records an architecture so that it can be logged, saved, and rebuilt.

In Keras, every layer reachable from an attribute is tracked, inside lists
and dictionaries too. Once the variables exist, the structure of the model
is fixed.

## Exercises

1. [code] **Unregistered modules.** Keras tracks a layer that the
   constructor assigns to an attribute, but not a `Dense` layer created
   *inside* `call`. Such a model runs without complaint. Write one, check
   `len(net.trainable_variables)` after calling it, explain what an
   optimizer would train, and explain what happens to the layer's weights
   between two calls.

2. **Widening a residual block.** `ResidualBlock` requires its input and
   output widths to agree. Suppose a block maps width $d$ to a larger width
   $d'$, so its body ends in a $d'$-wide layer and the skip path must also
   produce width $d'$. Give one way to build that skip path with
   parameters and one without. For each, count the extra parameters and
   state which output coordinates still receive the identity path.

3. [code] **Config boundaries.** Extend `MLPConfig` with a field
   `act: str = 'relu'` that selects the activation inside every
   `ResidualBlock`, and make `build` honor it.
    1. Build one model with `act='tanh'` and one from the default config,
       and compare their parameter shapes. Could a checkpoint that holds
       only the weights tell the two models apart? State what the answer
       implies for the config.
    1. A plain feed-forward block has the same `body` as `ResidualBlock`
       but no skip connection. Apply the same test to the choice between
       the two block types, and decide whether that choice belongs in the
       config or in code.

4. [code] **A module from scratch.** ● In plain Python and NumPy, without any
   framework base class, write a `Parameter` class that wraps an array and
   a minimal `Module` class whose `__setattr__` registers every assigned
   `Parameter` or `Module`. Give `Module` a `parameters()` method that
   recurses into the registered children. Write a `Linear` subclass of
   `Module` that holds a weight and a bias, build a two-layer MLP from two
   instances of it, and verify that `parameters()` returns all four arrays.
   Then store the two layers in a plain list, check what `parameters()`
   returns, and extend `Module` so that the list version finds all four
   arrays as well.

    *Adapted from CMU 10-714,
    [Homework 2](https://github.com/dlsyscourse/hw2), Question 2.*

5. [extended] **Depth family.** ● With `MLPConfig` and `build`, construct the
   residual MLP with `num_inputs=1`, `num_outputs=1`, and `num_blocks` set
   to 0, 1, and 4.
    1. Before training, list the layers that `build` stacks when
       `num_blocks=0` and state which functions of $x$ that model can
       represent.
    1. Train each variant with the squared loss, the same optimizer, and
       the same number of epochs on 1,000 points $x$ drawn uniformly from
       $[-3, 3]$ with targets $y = \sin(2x)$. Report a table of parameter
       count and final training loss against `num_blocks`, and explain the
       loss of the `num_blocks=0` variant from the first part.

    *Adapted from CMU 11-785,
    [Homework 1 Part 1](https://deeplearning.cs.cmu.edu/F23/document/homework/HW1/HW1P1_F23.pdf),
    Section 7.*

6. [code] **Lazy input width.** Build a `tf.keras.Sequential` holding
   `tf.keras.layers.Dense(256, activation='relu')` and
   `tf.keras.layers.Dense(10)`, and pass a batch with 20 features through
   it, then a batch with 30 features. Explain the outcome from what the
   first forward pass did to each layer. Then describe a model that accepts
   inputs of either width while sharing every layer after the first.

[Discussions](https://d2l.discourse.group/t/36752)